# 03 - Baseline B2: regresion logistica regularizada

B2 esta cerrado: la configuracion seleccionada es `C=0.1` en A, B y C. Cada ajuste usa todo el `train` propio del escenario y se mide en su `val`; `calib` y `test` quedan reservados.

Cada celda que entrena revisa primero SUS archivos en Kaggle. Si ya estan, los baja a `/content/resultados/b2/...` y no entrena. Si faltan, entrena solo lo que falta y lo publica. No hay interruptores que cambiar: basta ejecutar las celdas en orden.

In [ ]:
# PREPARACION DEL CUADERNO
# PROPOSITO: dejar este servidor de Colab listo SIN depender de otro cuaderno.
# - Clona el repo (o lo actualiza) e instala las librerias fijadas en requirements.txt.
# - Muestra nucleos de CPU y RAM. Para entrenar se espera CPU + RAM alta (8 nucleos, ~51 GiB).
# - Configura Kaggle: token de los secretos de Colab, de secrets.txt o pedido a mano (oculto).
# - Baja a /content/datos solo las tablas que falten en este servidor.
# Los RESULTADOS (CSV, JSON, .joblib) NO se guardan en Drive ni en el repo: viven en el dataset
# privado de Kaggle. Cada celda que entrena revisa SUS archivos en Kaggle: si ya estan, los baja
# a /content/resultados y no entrena; si faltan, entrena y los publica.
# Despues del git pull se "olvidan" los modulos ptcg_wp ya cargados para usar el codigo nuevo.
# ESPERADO: la linea de nucleos/RAM, "ya existe" o "bajado" para las 21 tablas y
# "Tablas listas | caracteristicas: 62".
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "04MainAIModelTrainingAndDevelopment"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

# Carpetas de trabajo: tablas en /content/datos y resultados en /content/resultados
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))

# Las 62 caracteristicas del modelo principal (sin ids de carta ni columnas meta)
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)
SEED = 2026
SCENARIOS = ("A", "B", "C")

# Trazabilidad que se guarda en cada registro.json nuevo
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

## Seleccion de C (historica, solo lectura)

La busqueda de `C` ya se hizo y no se repite aqui. Primero un piloto con 5000 partidas de `train` y nueve valores de `C`; luego una CV con todo `train` entre `C=0.1` y `C=1.0`, cuyas diferencias fueron minimas. Se eligio `C=0.1` (mas regularizacion con el mismo desempeno).

In [ ]:
# SELECCION DE C (SOLO LECTURA)
# PROPOSITO: mostrar los resumenes de la busqueda de C. No entrena: solo baja 2 CSV de Kaggle.
# ESPERADO: AUC casi identico para C >= 0.1 en el piloto y entre C=0.1 y C=1.0 con todo train.
RUTAS_SELECCION_B2 = ["b2/exploracion_muestra/b2_cv_summary.csv", "b2/seleccion_c/b2_cv_summary.csv"]
if store.listo(RUTAS_SELECCION_B2):
    piloto = pd.read_csv(RESULTADOS / RUTAS_SELECCION_B2[0])
    print("Piloto (5000 partidas de train): AUC promedio de CV por C")
    display(piloto.pivot(index="C", columns="escenario", values="auc").round(4))
    print("CV con todo train: C=0.1 contra C=1.0")
    display(pd.read_csv(RESULTADOS / RUTAS_SELECCION_B2[1]).round(4))
else:
    print("Los resumenes de la seleccion de C no estan en Kaggle.")

## Demostracion de fuga

Usa solo la configuracion finalista `C=0.1`. Con la misma muestra de 200 partidas de `train`, reajusta una regresion logistica para cada division: una por fila y otra por `game_id`. Usa LBFGS en una muestra pequena, mientras el B2 final usa SGD sobre todo `train`; por eso es un diagnostico de fuga, no una comparacion de desempeno. La diferencia observada no garantiza inflacion en todas las muestras.

In [ ]:
# DEMOSTRACION DE FUGA: DIVISION POR FILA CONTRA DIVISION POR PARTIDA
# PROPOSITO: por escenario, bajar de Kaggle la demostracion si ya existe; si falta, ejecutarla
# (~1 min por escenario) y publicarla.
# ESPERADO: la division por fila comparte ~200 partidas entre ajuste y evaluacion; la division
# por partida comparte 0. AUC por fila / por partida: A 0.775/0.740, B 0.806/0.815, C 0.788/0.676.
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.pending_experiments import leakage_comparison

ARCHIVOS_FUGA = ("metricas.csv", "solapamiento.csv", "config.json", "registro.json")
for escenario in SCENARIOS:
    base = f"b2/demo_fuga/escenario_{escenario}"
    print(f"--- Escenario {escenario} ---")

    def entrenar_fuga(escenario=escenario, base=base):
        salida = RESULTADOS / base
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES))
        leakage_comparison(stream, FEATURES, salida, seed=SEED, max_games=200, c_value=0.1)
        (salida / "registro.json").write_text(json.dumps(RUN_INFO, indent=2))

    store.asegurar([f"{base}/{n}" for n in ARCHIVOS_FUGA], entrenar_fuga,
                   f"B2 demostracion de fuga {escenario}")
    display(pd.read_csv(RESULTADOS / base / "metricas.csv").round(4))
    display(pd.read_csv(RESULTADOS / base / "solapamiento.csv"))

## Modelo final de B2 por escenario

Para cada escenario: si `b2_val_X.csv` y `b2_X_C0.1.joblib` ya estan en Kaggle, se bajan y se verifican sus metadatos (C, caracteristicas, 600 epocas, scikit-learn 1.7.2 y uso de todo `train`). Si falta alguno, se ajusta `C=0.1` con todo `train` del escenario (varias horas), se mide en `val` y se publica. La corrida original no alcanzo la tolerancia de gradiente tras 600 epocas; esa limitacion se conserva.

In [ ]:
# B2 FINAL: BAJAR DE KAGGLE O AJUSTAR C=0.1 CON TODO TRAIN, Y VERIFICAR
# PROPOSITO: dejar en /content/resultados/b2/final/escenario_X/ el modelo y sus metricas de val.
# - Si ambos archivos estan en Kaggle: se bajan y NO se entrena.
# - Si falta alguno: se ajusta solo ese escenario y se publica al terminarlo.
# ESPERADO: por escenario "train completo: 33066 (A) / 27643 (B) / 33013 (C) partidas" y
# AUC val 0.7607 (A), 0.7281 (B), 0.7603 (C).
import joblib
from ptcg_wp.b2 import fit_b2_candidates_val
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.finalist_evidence import selected_train_count, verify_full_train

for escenario in SCENARIOS:
    base = f"b2/final/escenario_{escenario}"
    csv_val, modelo = f"{base}/b2_val_{escenario}.csv", f"{base}/b2_{escenario}_C0.1.joblib"
    print(f"--- Escenario {escenario} ---")

    def entrenar_b2(escenario=escenario, base=base):
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES), batch_size=32768)
        fit_b2_candidates_val(stream, FEATURES, (0.1,), max_iter=600, train_games=None,
                              seed=SEED, cache_dir="/content", output_dir=RESULTADOS / base,
                              eta0=0.05, tolerance=1e-3, verbose=True)

    store.asegurar([csv_val, modelo], entrenar_b2, f"B2 final C=0.1 escenario {escenario}")

    # Verificacion de metadatos del modelo contra su CSV de val
    item = joblib.load(RESULTADOS / modelo)
    assert item.get("escenario") == escenario and float(item["C"]) == 0.1
    assert list(item["features"]) == list(FEATURES)
    assert item.get("max_epocas") == 600 and item.get("version_sklearn") == "1.7.2"
    esperado = selected_train_count(RESULTADOS / csv_val, escenario, "B2")
    verify_full_train(item, escenario, "B2", esperado)
    print(f"B2 {escenario} | train completo: {esperado} partidas | SHA256: "
          f"{store.sha256(RESULTADOS / modelo)[:16]}... | convergencia: {item.get('convergencia')}")
    val = pd.read_csv(RESULTADOS / csv_val)
    display(val.loc[val["etapa"].eq("global")].round(4))